## Caso Práctico – Análisis de datos de la copa mundial de fútbol con Apache Spark

## 1.- Instalar entorno de Spark: Configuración inicial del entorno

1.1. Instalar Apache Spark y Java en el entorno de ejecución

1.2. Crear las variables de entorno necesarias

## 1.3. Crear la SparkSession con el nombre "MundialAnalysis" y el SparkContext.

In [ ]:
# esto la creacion del entorno de spark y la creacion de la sesion de spark
## luego se listan los archivos disponibles en el directorio "Formativa" y se muestran las variables de entorno relacionadas con Spark y Java.

import os
import sys
import subprocess
from pathlib import Path

import pyspark
import findspark

java_home = subprocess.check_output(["/usr/libexec/java_home"]).decode().strip()
os.environ["JAVA_HOME"] = java_home
os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["SPARK_HOME"] = os.path.dirname(pyspark.__file__)

findspark.init()

sc = pyspark.SparkContext.getOrCreate()
spark = (
    pyspark.sql.SparkSession.builder
    .master("local[*]")
    .appName("MundialAnalysis")
    .getOrCreate()
)

data_dir = Path.cwd() / "Formativa"

print("JAVA_HOME =", os.environ["JAVA_HOME"])
print("SPARK_HOME =", os.environ["SPARK_HOME"])
print("SparkSession creada:", spark.conf.get("spark.app.name"))
print("SparkContext activo:", sc)
print("Archivos disponibles en Formativa:")
for archivo in sorted(data_dir.iterdir()):
    print("-", archivo.name)

JAVA_HOME = /Library/Java/JavaVirtualMachines/jdk-18.0.1.1.jdk/Contents/Home
SPARK_HOME = /Users/sauriomac/Documents/spark-semana2/.venv/lib/python3.10/site-packages/pyspark
SparkSession creada: MundialAnalysis
SparkContext activo: <SparkContext master=local[*] appName=pyspark-shell>
Archivos disponibles en Formativa:
- equipos.csv
- estadios.csv
- jugadores.csv
- partidos.csv
- torneos.json


# 2.- RDDs - Creación y unión (ID 3.1)

## 2.1. Crear un RDD llamado jugador1 con 6 particiones, leyendo el archivo jugadores.csv.

In [ ]:
## se carga el archivo "jugadores.csv" en un RDD llamado jugador1, utilizando 6 particiones para distribuir los datos de manera eficiente.

jugador1 = sc.textFile("Formativa/jugadores.csv", 6)

## 2.2. Crear un segundo RDD llamado jugador2 con 6 particiones, leyendo nuevamente el archivo jugadores.csv.

In [ ]:
## se carga el archivo "jugadores.csv" en un RDD llamado jugador2, utilizando 6 particiones para distribuir los datos de manera eficiente.

jugador2 = sc.textFile("Formativa/jugadores.csv", 6)

## 2.3. Crear un nuevo RDD llamado jugadorTotal que contenga la unión de los RDDs jugador1 y jugador2.

In [ ]:
# 2.3. Crear un nuevo RDD llamado jugadorTotal que contenga la unión de los RDDs jugador1 y jugador2.
jugadorTotal = jugador1.union(jugador2)

## 2.4. Mostrar la cantidad de registros contenidos en jugadorTotal.

In [ ]:
## se muestra el número total de registros en el RDD jugadorTotal utilizando el método count(). Esto permite conocer la cantidad de datos combinados de los dos RDDs originales.

jugadorTotal.count()

162

## 2.5. Convertir el RDD jugadorTotal en un DataFrame llamado jugadores con las columnas solicitadas.

In [91]:
## se conevierte el RDD jugadorTotal en un DataFrame llamado jugadores, utilizando la función Row para definir la estructura de los datos y especificando los nombres de las columnas.
## y luego se muestra el contenido del DataFrame jugadores utilizando el método show() para ver las primeras filas del DataFrame.

filas = jugadorTotal.map(lambda line: line.split(",")).filter(lambda fila: fila[0] != "jugador_id")

datos = filas.map(lambda fila: Row(
    jugador_id=fila[0],
    nombre=fila[1],
    apellido=fila[2],
    edad=fila[3],
    altura=fila[4],
    peso=fila[5],
    posicion=fila[6],
    equipo_id=fila[7],
))

jugadores = datos.toDF([
    "jugador_id",
    "nombre",
    "apellido",
    "edad",
    "altura",
    "peso",
    "posicion",
    "equipo_id",
])


## 2.6. Mostrar el esquema y los tipos de datos utilizables del DataFrame jugadores.

In [ ]:
## aca se muestra el esquema del DataFrame jugadores y se muestran las primeras filas del DataFrame utilizando el método show(). Esto permite verificar la estructura de los datos y obtener una vista previa de su contenido.
## y luego se muestra el contenido del DataFrame jugadores utilizando el método show() para ver las primeras filas del DataFrame.

jugadores.printSchema()
jugadores.show()

root
 |-- jugador_id: string (nullable = true)
 |-- nombre: string (nullable = true)
 |-- apellido: string (nullable = true)
 |-- edad: string (nullable = true)
 |-- altura: string (nullable = true)
 |-- peso: string (nullable = true)
 |-- posicion: string (nullable = true)
 |-- equipo_id: string (nullable = true)

+----------+---------+---------+----+------+----+-------------+---------+
|jugador_id|   nombre| apellido|edad|altura|peso|     posicion|equipo_id|
+----------+---------+---------+----+------+----+-------------+---------+
|         1|    María| González|  29|   166|  73|      Portero|       10|
|         2|    Diego|    López|  35|   195|  76|      Defensa|        8|
|         3|   Isabel|    Pérez|  36|   170|  83|    Delantero|       11|
|         4|      Ana|    López|  22|   185|  69|    Delantero|        4|
|         5| Patricia|   Torres|  27|   191|  69|Mediocampista|       15|
|         6|    Sofía|     Díaz|  26|   191|  95|      Defensa|        9|
|         7|   Mi

## Parte 3: RDDs - Transformaciones (ID 3.2)

### 3.1. Crear un RDD llamado MayorEdad que permita filtrar jugadores mayores de 30 años.

In [ ]:
## se crea un nuevo DataFrame llamado MayorEdad que contiene solo los jugadores mayores de 30 años

MayorEdad = jugadores.filter(jugadores.edad > 30)

In [ ]:
# y luego se muestra el contenido del DataFrame MayorEdad utilizando el método show() para ver las primeras filas del DataFrame.

MayorEdad.show()

+----------+--------+---------+----+------+----+-------------+---------+
|jugador_id|  nombre| apellido|edad|altura|peso|     posicion|equipo_id|
+----------+--------+---------+----+------+----+-------------+---------+
|         2|   Diego|    López|  35|   195|  76|      Defensa|        8|
|         3|  Isabel|    Pérez|  36|   170|  83|    Delantero|       11|
|         8| Roberto| Martínez|  33|   177|  83|    Delantero|       12|
|        10|  Carmen|   Castro|  36|   179|  82|Mediocampista|       14|
|        13|  Camila|    López|  33|   193|  89|Mediocampista|       13|
|        18| Antonio|    Gómez|  37|   169|  67|      Defensa|       15|
|        21|Patricia|   Torres|  33|   180|  83|    Delantero|        4|
|        23|  Miguel|   Torres|  35|   177|  82|Mediocampista|        2|
|        24|Fernando| González|  31|   193|  80|      Portero|        9|
|        27|   Sofía|  Jiménez|  38|   172|  90|    Delantero|        7|
|        30|  Miguel|  Jiménez|  33|   178|  77|   

### 3.2. Crear un RDD llamado Jugadores_Defensa que muestre solo los jugadores cuya posición sea "Defensa".

In [ ]:
## se crea un nuevo DataFrame llamado jugadores_defensas que contiene solo los jugadores cuya posición es "Defensa"
## luego se muestra el contenido del DataFrame jugadores_defensas utilizando el método show().
 
jugadores_defensas = jugadores.filter(jugadores.posicion == "Defensa")
jugadores_defensas.show()

+----------+---------+---------+----+------+----+--------+---------+
|jugador_id|   nombre| apellido|edad|altura|peso|posicion|equipo_id|
+----------+---------+---------+----+------+----+--------+---------+
|         2|    Diego|    López|  35|   195|  76| Defensa|        8|
|         6|    Sofía|     Díaz|  26|   191|  95| Defensa|        9|
|         9|    Pedro|    López|  25|   193|  81| Defensa|        8|
|        14| Patricia|  Ramírez|  28|   183|  67| Defensa|       14|
|        16|   Carmen| González|  27|   179|  75| Defensa|        9|
|        18|  Antonio|    Gómez|  37|   169|  67| Defensa|       15|
|        38|   Camila|Rodríguez|  38|   177|  84| Defensa|        3|
|        45| Gabriela|   Castro|  27|   180|  78| Defensa|        4|
|        49|  Roberto|  Morales|  31|   176|  81| Defensa|        4|
|        51|Valentina|   Flores|  27|   167|  71| Defensa|       10|
|        52|  Roberto|Rodríguez|  33|   178|  71| Defensa|       13|
|        57|Valentina|    López|  

## 3.3. Convertir a mayúsculas todas las palabras (nombre y apellido) del RDD jugadorTotal.

In [ ]:
## se convierte a mayusculas el nombre y apellido de los jugadores
## por medio de la funcion map y lambda se hace la conversion a mayusculas
## luego se hace un collect para mostrar los resultados

jugadoresMayusculas = jugadorTotal.map(
    lambda x: (
        x.split(",")[1].upper(),
        x.split(",")[2].upper()
    )
)

jugadoresMayusculas.collect()

[('NOMBRE', 'APELLIDO'),
 ('MARÍA', 'GONZÁLEZ'),
 ('DIEGO', 'LÓPEZ'),
 ('ISABEL', 'PÉREZ'),
 ('ANA', 'LÓPEZ'),
 ('PATRICIA', 'TORRES'),
 ('SOFÍA', 'DÍAZ'),
 ('MIGUEL', 'RODRÍGUEZ'),
 ('ROBERTO', 'MARTÍNEZ'),
 ('PEDRO', 'LÓPEZ'),
 ('CARMEN', 'CASTRO'),
 ('VALENTINA', 'ÁLVAREZ'),
 ('LUIS', 'DÍAZ'),
 ('CAMILA', 'LÓPEZ'),
 ('PATRICIA', 'RAMÍREZ'),
 ('ROBERTO', 'PÉREZ'),
 ('CARMEN', 'GONZÁLEZ'),
 ('MIGUEL', 'RODRÍGUEZ'),
 ('ANTONIO', 'GÓMEZ'),
 ('MARÍA', 'RIVERA'),
 ('VALENTINA', 'VARGAS'),
 ('PATRICIA', 'TORRES'),
 ('ISABEL', 'MORALES'),
 ('MIGUEL', 'TORRES'),
 ('FERNANDO', 'GONZÁLEZ'),
 ('GABRIELA', 'RAMÍREZ'),
 ('MARÍA', 'MARTÍNEZ'),
 ('SOFÍA', 'JIMÉNEZ'),
 ('PEDRO', 'LÓPEZ'),
 ('LUIS', 'MARTÍNEZ'),
 ('MIGUEL', 'JIMÉNEZ'),
 ('MARÍA', 'ROMERO'),
 ('MIGUEL', 'GARCÍA'),
 ('MIGUEL', 'RAMÍREZ'),
 ('MIGUEL', 'RAMÍREZ'),
 ('JUAN', 'GÓMEZ'),
 ('PATRICIA', 'MARTÍNEZ'),
 ('LAURA', 'RODRÍGUEZ'),
 ('CAMILA', 'RODRÍGUEZ'),
 ('PEDRO', 'CASTRO'),
 ('JOSÉ', 'SÁNCHEZ'),
 ('LAURA', 'MORALES'),
 ('DIEGO', 

## Parte 4: DataFrames - Creación e integración (ID 3.1)


### 4.1. Crear los DataFrames Equipos, Partidos, Estadios y Torneos a partir de los archivos equipos.csv, partidos.csv, estadios.csv y torneos.json.


In [ ]:
# creacion de un nuevo DataFrame llamado Equipos desde un archivo CSV llamado "equipos.csv" que se encuentra en el directorio "Formativa".

Equipos = ('Formativa/equipos.csv')

# equipos desde un archivo CSV llamado "equipos.csv" que se encuentra en el directorio "Formativa".

df_equipos = spark.read.csv(Equipos,sep=',',header=True, inferSchema=True)
# se muestra el contenido del DataFrame df_equipos utilizando el método show() para ver las primeras filas del DataFrame.
# Esto permite verificar la estructura de los datos y obtener una vista previa de su contenido.

df_equipos.show()

+---------+----------+-------------+------------+
|equipo_id|    nombre|confederacion|ranking_fifa|
+---------+----------+-------------+------------+
|        1| Argentina|     CONMEBOL|          30|
|        2|    Brasil|     CONMEBOL|          39|
|        3|    España|         UEFA|          23|
|        4|  Alemania|         UEFA|          50|
|        5|   Francia|         UEFA|          23|
|        6|    Italia|         UEFA|           7|
|        7|   Uruguay|     CONMEBOL|           4|
|        8|Inglaterra|         UEFA|          19|
|        9|  Portugal|         UEFA|          29|
|       10|   Holanda|         UEFA|          42|
|       11|    México|     CONCACAF|          42|
|       12|  Colombia|     CONMEBOL|          45|
|       13|     Chile|     CONMEBOL|          29|
|       14|   Bélgica|         UEFA|          41|
|       15|   Croacia|         UEFA|          33|
+---------+----------+-------------+------------+



In [111]:
# creacion de un nuevo DataFrame llamado Partidos desde un archivo CSV llamado "partidos.csv" que se encuentra en el directorio "Formativa".

Partidos= ('Formativa/partidos.csv')

# partidos desde un archivo CSV llamado "partidos.csv" que se encuentra en el directorio "Formativa".

df_partidos = spark.read.csv(Partidos,sep=',',header=True, inferSchema=True)
# se muestra el contenido del DataFrame df_partidos utilizando el método show() para ver las primeras filas del DataFrame.
# Esto permite verificar la estructura de los datos y obtener una vista previa de su contenido.

df_partidos.show()

+----------+---------------+-------------------+-----------+---------------+----------+---------+----------------+
|partido_id|equipo_local_id|equipo_visitante_id|goles_local|goles_visitante|estadio_id|torneo_id|            fase|
+----------+---------------+-------------------+-----------+---------------+----------+---------+----------------+
|         1|              9|                  5|          2|              3|         3|        3|       Semifinal|
|         2|             11|                  9|          2|              1|         7|        2|Octavos de Final|
|         3|             11|                  5|          5|              3|         2|        2|       Semifinal|
|         4|              1|                  9|          3|              3|         8|        1|       Semifinal|
|         5|              5|                  3|          3|              4|         1|        1|Octavos de Final|
|         6|              6|                 14|          0|              1|    

In [97]:
# creacion de un nuevo DataFrame llamado Estadios desde un archivo CSV llamado "estadios.csv" que se encuentra en el directorio "Formativa".

Estadios = ('Formativa/estadios.csv')

# estadios desde un archivo CSV llamado "estadios.csv" que se encuentra en el directorio "Formativa".

df_estadios = spark.read.csv(Estadios,sep=',',header=True, inferSchema=True)
# se muestra el contenido del DataFrame df_estadios utilizando el método show() para ver las primeras filas del DataFrame.
# Esto permite verificar la estructura de los datos y obtener una vista previa de su contenido.

df_estadios.show()

+----------+-----------------+----------------+----------+---------+
|estadio_id|           nombre|          ciudad|      pais|capacidad|
+----------+-----------------+----------------+----------+---------+
|         1|   Estadio Azteca|Ciudad de México|    México|    87523|
|         2|         Maracaná|  Río de Janeiro|    Brasil|    78838|
|         3|         Camp Nou|       Barcelona|    España|    99354|
|         4|          Wembley|         Londres|Inglaterra|    90000|
|         5|    Allianz Arena|          Múnich|  Alemania|    75000|
|         6|         San Siro|           Milán|    Italia|    80018|
|         7|Santiago Bernabéu|          Madrid|    España|    81044|
|         8|       Monumental|    Buenos Aires| Argentina|    83196|
|         9|           Lusail|          Lusail|     Qatar|    88966|
|        10|       Centenario|      Montevideo|   Uruguay|    60235|
+----------+-----------------+----------------+----------+---------+



In [106]:
# creacion de un nuevo DataFrame llamado Partidos desde un archivo CSV llamado "torneos.json" que se encuentra en el directorio "Formativa".

Torneos= ('Formativa/torneos.json')

# partidos desde un archivo CSV llamado "torneos.json" que se encuentra en el directorio "Formativa".

df_torneos = spark.read.option("multiline", "true").json(Torneos)
# se muestra el contenido del DataFrame df_partidos utilizando el método show() para ver las primeras filas del DataFrame.
# Esto permite verificar la estructura de los datos y obtener una vista previa de su contenido.

# print('DataFrame creado desde JSON:', df_torneos)
df_torneos.show()


+----+---------+--------------------+---------+----------+---------+
|anio|  campeon|              nombre|pais_sede|subcampeon|torneo_id|
+----+---------+--------------------+---------+----------+---------+
|2018|  Francia|Copa Mundial FIFA...|    Rusia|   Croacia|        1|
|2014| Alemania|Copa Mundial FIFA...|   Brasil| Argentina|        2|
|2022|Argentina|Copa Mundial FIFA...|    Qatar|   Francia|        3|
+----+---------+--------------------+---------+----------+---------+



### 4.2. Realizar la optimización de los DataFrames creados (aplicar .cache() o .persist() según corresponda).


### 4.3. Crear un nuevo DataFrame llamado mundial_completo que contenga los datos de todos los DataFrames ( jugadores, equipos, partidos, estadios y torneos) mediante operaciones de join.

In [121]:
# creacion de un nuevo dataframe llamado mundial_completo que contiene todos los dataframes: jugadores,equipos, partidos y torneos mediante join
# usare un alias por cada dataframe para evitar conflictos de nombres de columnas y facilitar la referencia a las columnas en las operaciones posteriores.

dj = jugadores.alias("j")
de = df_equipos.alias("e")
dp = df_partidos.alias("p")
dt = df_torneos.alias("t")
dest = df_estadios.alias("est")

# se crean dos DataFrames intermedios, partidos_locales y partidos_visitantes, que contienen la información de los partidos jugados por los equipos locales y visitantes, respectivamente.
partidos_locales = df_partidos.select(
    "*",
    df_partidos["equipo_local_id"].alias("equipo_id")
)

partidos_visitantes = df_partidos.select(
    "*",
    df_partidos["equipo_visitante_id"].alias("equipo_id")
)

# se crea un DataFrame llamado participaciones que contiene la unión de los DataFrames partidos_locales y partidos_visitantes, utilizando la función unionByName para combinar las filas de ambos DataFrames basándose en los nombres de las columnas.
participaciones = partidos_locales.unionByName(
    partidos_visitantes
)

# se muestran los nombres de las columnas de cada DataFrame para verificar la estructura de los datos antes de realizar la unión final en el DataFrame mundial_completo.
print("Jugadores:", dj.columns)
print("Equipos:", de.columns)
print("Partidos:", dp.columns)
print("Torneos:", dt.columns)
print("Estadios:", dest.columns)

# se crea un DataFrame llamado mundial_completo que contiene la unión de los DataFrames jugadores, df_equipos, participaciones, df_torneos y df_estadios, utilizando la función join para combinar las filas de los DataFrames basándose en las columnas equipo_id, torneo_id y estadio_id.
mundial_completo = (
    jugadores
    .join(df_equipos, on="equipo_id", how="inner")
    .join(participaciones, on="equipo_id", how="inner")
    .join(df_torneos, on="torneo_id", how="inner")
    .join(df_estadios, on="estadio_id", how="inner")
)
# se muestra el contenido del DataFrame mundial_completo utilizando el método show() para ver las primeras filas del DataFrame.
mundial_completo.show(truncate=False)

Jugadores: ['jugador_id', 'nombre', 'apellido', 'edad', 'altura', 'peso', 'posicion', 'equipo_id']
Equipos: ['equipo_id', 'nombre', 'confederacion', 'ranking_fifa']
Partidos: ['partido_id', 'equipo_local_id', 'equipo_visitante_id', 'goles_local', 'goles_visitante', 'estadio_id', 'torneo_id', 'fase']
Torneos: ['anio', 'campeon', 'nombre', 'pais_sede', 'subcampeon', 'torneo_id']
Estadios: ['estadio_id', 'nombre', 'ciudad', 'pais', 'capacidad']
+----------+---------+---------+----------+------+--------+----+------+----+--------+----------+-------------+------------+----------+---------------+-------------------+-----------+---------------+----------------+----+---------+----------------------+---------+----------+-----------------+----------------+----------+---------+
|estadio_id|torneo_id|equipo_id|jugador_id|nombre|apellido|edad|altura|peso|posicion|nombre    |confederacion|ranking_fifa|partido_id|equipo_local_id|equipo_visitante_id|goles_local|goles_visitante|fase            |anio|cam

### Parte 5: Paralelismo (ID 3.4)
5.1. Paralelizar el DataFrame mundial_completo a 5 particiones.


In [129]:
# paralelizar el dataframe mundial_completo para crear un RDD llamado rdd_mundial_completo, utilizando el método rdd del DataFrame. Esto permite trabajar con los datos en un formato distribuido y realizar operaciones de transformación y acción en paralelo.
rdd_mundial_completo = mundial_completo.rdd

print("Número de particiones del RDD rdd_mundial_completo:", rdd_mundial_completo.getNumPartitions())
# particionar el RDD rdd_mundial_completo en 5 particiones utilizando el método repartition(). Esto permite distribuir los datos de manera más eficiente y mejorar el rendimiento de las operaciones posteriores.
new_rdd_mundial_completo = rdd_mundial_completo.repartition(5)



Número de particiones del RDD rdd_mundial_completo: 12


### 5.2. Verificar el número de particiones del DataFrame resultante.
Caso Práctico – Análisis de datos de la copa mundial de fútbol con Apache Spark


In [139]:
# verificar el numero de particiones del RDD rdd_mundial_completo utilizando el método getNumPartitions() para obtener el número de particiones actuales del RDD.
print("Número de particiones del RDD rdd_mundial_completo:", new_rdd_mundial_completo.getNumPartitions())

Número de particiones del RDD rdd_mundial_completo: 5


### Parte 6: Inspección del dataFrame (ID 3.1)
6.1. Mostrar la cantidad de filas, el tipo de datos y el esquema del DataFrame
mundial_completo.


In [138]:
# mostrar la cantidade de filas, el tipo de datos y el esquema del dataframe mundial_completo utilizando los métodos count(), dtypes y printSchema() respectivamente. Esto permite obtener información sobre la cantidad de registros, los tipos de datos de cada columna y la estructura del DataFrame mundial_completo.
print("Cantidad de filas en el DataFrame mundial_completo:", mundial_completo.count())

tipo_de_col = mundial_completo.dtypes

print(
    "Tipos de datos de cada columna:\n" +
    "\n".join(
        f"- {columna}: {tipo}"
        for columna, tipo in mundial_completo.dtypes
    )
)
print("=========================================================")

print("Esquema del DataFrame mundial_completo:")
mundial_completo.printSchema()




Cantidad de filas en el DataFrame mundial_completo: 2108
Tipos de datos de cada columna:
- estadio_id: int
- torneo_id: int
- equipo_id: string
- jugador_id: string
- nombre: string
- apellido: string
- edad: string
- altura: string
- peso: string
- posicion: string
- nombre: string
- confederacion: string
- ranking_fifa: int
- partido_id: int
- equipo_local_id: int
- equipo_visitante_id: int
- goles_local: int
- goles_visitante: int
- fase: string
- anio: bigint
- campeon: string
- nombre: string
- pais_sede: string
- subcampeon: string
- nombre: string
- ciudad: string
- pais: string
- capacidad: int
Esquema del DataFrame mundial_completo:
root
 |-- estadio_id: integer (nullable = true)
 |-- torneo_id: integer (nullable = true)
 |-- equipo_id: string (nullable = true)
 |-- jugador_id: string (nullable = true)
 |-- nombre: string (nullable = true)
 |-- apellido: string (nullable = true)
 |-- edad: string (nullable = true)
 |-- altura: string (nullable = true)
 |-- peso: string (nullab

### Parte 7: Columnas calculadas (ID 3.3)
7.1. Crear una columna calculada llamada IMC que determine el Índice de Masa
Corporal de cada jugador usando la fórmula:
IMC = peso / (altura/100)²
7.2. Crear una columna calculada llamada Categoria_Edad que determine la categoría
del jugador:
• "Joven" si edad < 25
• "Experimentado" si edad entre 25 y 32
• "Veterano" si edad > 32
7.3. Crear una columna calculada llamada Resultado_Partido que determine el
resultado del partido:
• "Victoria Local" si goles_local > goles_visitante
• "Victoria Visitante" si goles_visitante > goles_local
• "Empate" si goles_local == goles_visitante

### Parte 8: Agregaciones con Spark SQL (ID 3.3)
Registre el DataFrame mundial_completo como una vista temporal llamada "Mundial"
y utilice Spark SQL para responder:
## 8.1. ¿Cuántos jugadores hay por equipo? (Conteo de jugadores agrupado por nombre
de equipo)
## 8.2. ¿Cuál es la edad promedio, mínima y máxima de los jugadores, agrupado por
posición?
## 8.3. ¿Cuál es la altura promedio, mínima y máxima de los jugadores, agrupado por
confederación?
## 8.4. ¿Cuántos partidos se jugaron en cada fase del torneo?
## 8.5. ¿Cuál es el total de goles anotados por cada equipo (como local y visitante)?
Caso Práctico – Análisis de datos de la copa mundial de fútbol con Apache Spark
8
## 8.6. ¿Cuál es el promedio de goles por partido en cada torneo?
## 8.7. ¿Cuál es la capacidad promedio de los estadios agrupados por país?
## 8.8. Utilizando CASE WHEN, cree una consulta SQL que agregue una columna
Clasificacion_IMC con las siguientes categorías:
• "Bajo peso" si IMC < 20
• "Normal" si IMC entre 20 y 25
• "Sobrepeso" si IMC > 25
## 8.9. ¿Cuál es el número de victorias, derrotas y empates de cada equipo?
## 8.10. Utilizando HAVING, encuentre los equipos que tienen un promedio de edad
mayor a 28 años.

## Parte 9: Transformaciones avanzadas (ID 3.2)
9.1. Filtrar el DataFrame para mostrar solo los partidos de la fase "Final".
9.2. Seleccionar solo las columnas: nombre_jugador, apellido, posicion,
nombre_equipo.
9.3. Ordenar el DataFrame por edad de mayor a menor.
9.4. Mostrar los 10 jugadores más altos del torneo.